# Masterclass: Tool Engineering & Custom Tool Creation in LangChain & LangGraph

Welcome to the comprehensive guide on **Tool Engineering**. In Agentic AI systems, **Tool Calling** enables an LLM to act as a **decision maker**—invoking external APIs, databases, calculators, web search tools, or custom functions when it requires data beyond its pre-trained knowledge.

### Conceptual Architecture: Node vs. Tool Execution

- **Graph Node**: Executed **deterministically by the graph workflow** engine.
- **Tool**: Executed **dynamically when chosen by the LLM**.

```text
User Input
    │
    ▼
[LLM Decision Engine] ──► Does it need external data/actions?
    │
    ├─────────────────────────────┐
    ▼                             ▼
 (NO) Direct Answer            (YES) Tool Call Request (Name + Args JSON)
                                  │
                                  ▼
                           [Tool Executor (Python Function)]
                                  │
                                  ▼
                           Tool Output Result
                                  │
                                  ▼
                        [LLM Generates Final Answer]
```

### Comprehensive Tool Creation Patterns Matrix

| Pattern | Technique | Primary Use Case |
|---|---|---|
| **1** | `@tool` Decorator | Quick conversion of Python functions into tools |
| **2** | `@tool("custom_name", return_direct=True)` | Overriding tool names and controlling direct response return |
| **3** | `@tool(args_schema=PydanticModel)` | Enforcing strict type validation and JSON schemas |
| **4** | `@tool(parse_docstring=True)` | Auto-extracting input field descriptions from Google/NumPy docstrings |
| **5** | Asynchronous Tools (`async def`) | Non-blocking I/O operations (HTTP requests, DB queries) |
| **6** | `ToolRuntime` Context | Accessing active LangGraph `state` dynamically inside tool functions |
| **7** | `Tool.from_function()` | Constructing single-input tools without function decorators |
| **8** | `StructuredTool.from_function()` | Constructing multi-input tools imperatively with Pydantic schemas |
| **9** | Subclassing `BaseTool` | Enterprise object-oriented tool creation with full lifecycle control |

## 1. Setup & Environment Verification
Load environment variables from `.env` and verify API key availability.

In [ ]:
# Verify setup and check availability of API keys in environment
from dotenv import load_dotenv
import os
load_dotenv()

print("Setup loaded.")
print("GROQ_API_KEY available:", bool(os.getenv("GROQ_API_KEY")))
print("GOOGLE_API_KEY available:", bool(os.getenv("GOOGLE_API_KEY")))
print("TAVILY_API_KEY available:", bool(os.getenv("TAVILY_API_KEY")))

Setup loaded.
GROQ_API_KEY available: True
GOOGLE_API_KEY available: True
TAVILY_API_KEY available: True


## 2. Tool Creation via `@tool` Decorator

The `@tool` decorator is the primary way to convert Python functions into LangChain/LangGraph tools. LangChain automatically inspects function names, type hints, and docstrings to construct the tool metadata.

### 2.1 Basic `@tool` Decorator & Inspection
Inspect automatic tool attributes: `.name`, `.description`, and `.args`.

In [2]:
# Import @tool decorator from langchain_core.tools
from langchain_core.tools import tool

In [3]:
# Define basic calculator function with @tool decorator
@tool
def add_basic(a: int, b:int) -> int:
    """Add two numbers"""
    return a + b

In [4]:
# Inspect automatically inferred tool name
print("Name:", add_basic.name)

Name: add_basic


In [5]:
# Inspect automatically extracted tool description from docstring
print("Description:", add_basic.description)

Description: Add two numbers


In [6]:
# Inspect inferred tool input arguments and types
print("Args:", add_basic.args)

Args: {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [7]:
# Invoke tool using dictionary input format
add_basic.invoke({"a": 20, "b": 30})

50

In [8]:
# Execute tool invocation and store/print result
result = add_basic.invoke({"a": 20, "b": 30})
print("Execution result:", result)

Execution result: 50


### 2.2 Custom Tool Names & Execution Flags
Customize tool metadata using `@tool("custom_name", description=..., return_direct=True)`. Setting `return_direct=True` returns tool results directly to the user without sending them back to the LLM.

In [10]:
# Define tool with custom name override
@tool("calculator")
def add_with_custom_name(a:int, b:int) -> int:
    """Add two numbers."""
    return a + b

In [11]:
# Verify custom tool name and execute invocation
print("Tool name:", add_with_custom_name.name)
print("Execution result:", add_with_custom_name.invoke({"a": 10, "b": 15}))

Tool name: calculator
Execution result: 25


In [12]:
# Define tool with explicit name, description, and return_direct flag
@tool(
    "multiply_numbers",
    description="Multiply two integers and return the result.",
    return_direct=False,
)
def multiply_with_options(a, b):
    return a * b

In [13]:
# Inspect custom tool attributes and execute invocation
print("Name:", multiply_with_options.name)
print("Description:", multiply_with_options.description)
print("return_direct:", multiply_with_options.return_direct)
print("Execution result:", multiply_with_options.invoke({"a": 6, "b": 7}))

Name: multiply_numbers
Description: Multiply two integers and return the result.
return_direct: False
Execution result: 42


In [22]:
# we get result beacuse we have not enforced a schema
print("Execution result:", multiply_with_options.invoke({"a": "areeb", "b": 7}))

Execution result: areebareebareebareebareebareebareeb


### 2.3 Strict Input Validation with Pydantic `args_schema`
Use a Pydantic `BaseModel` to enforce strict type checking, default values, and parameter descriptions. Invalid inputs raise a `ValidationError` instead of executing bad code.

In [16]:
# Import Pydantic BaseModel and Field for strict schema enforcement
from pydantic import BaseModel, Field, ValidationError

In [17]:
# Define Pydantic schema for input argument validation
class CalculatorInputTest(BaseModel):
    a: int = Field(description="First integer")
    b: int = Field(description="Second integer")

In [18]:
# Bind Pydantic args_schema to tool
@tool(args_schema=CalculatorInputTest)
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

In [19]:
# Inspect generated Pydantic JSON schema
print("Schema:", multiply.args_schema.model_json_schema())

Schema: {'properties': {'a': {'description': 'First integer', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'Second integer', 'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'CalculatorInputTest', 'type': 'object'}


In [20]:
# Execute tool with valid integer arguments
print("Execution result:", multiply.invoke({"a": 8, "b": 9}))

Execution result: 72


In [21]:
# it wont work with string as input as we have implemented enforce schema
multiply.invoke({"a": "areeb", "b": 9})

ValidationError: 1 validation error for CalculatorInputTest
a
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='areeb', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

### 2.4 Automatic Schema Generation with `@tool(parse_docstring=True)`
Automatically generate Pydantic argument schemas directly from Google or NumPy style docstring `Args:` sections without writing manual schema classes.

In [23]:
# Define tool with parse_docstring=True to auto-generate schema from docstring Args:
@tool(parse_docstring=True)
def search_with_docstring(query: str, limit: int) -> str:
    """Search documents.

    Args:
        query: Search query entered by the user.
        limit: Maximum number of results.
    """
    return f"Searching for '{query}' with limit={limit}"

In [24]:
# Inspect auto-generated JSON schema from docstring
print("Args schema:")
print(search_with_docstring.args_schema.model_json_schema())

Args schema:
{'description': 'Search documents.', 'properties': {'query': {'description': 'Search query entered by the user.', 'title': 'Query', 'type': 'string'}, 'limit': {'description': 'Maximum number of results.', 'title': 'Limit', 'type': 'integer'}}, 'required': ['query', 'limit'], 'title': 'search_with_docstring', 'type': 'object'}


In [25]:
# Execute invocation on docstring-parsed tool
print("Execution result:", search_with_docstring.invoke({"query": "LangGraph memory","limit": 3,}))

Execution result: Searching for 'LangGraph memory' with limit=3


### 2.5 Asynchronous Tools (`async def` & `ainvoke`)
Define non-blocking asynchronous tools using `async def` and execute them using `await tool.ainvoke(...)`.

In [26]:
# Import asyncio for asynchronous execution
import asyncio

In [28]:
# Define async tool function using async def
@tool
async def get_data_async(url: str) -> str:
    """Fetch data asynchronously"""
    await asyncio.sleep(0.1)
    return f"Data from {url}"

In [29]:
# Execute async tool using await tool.ainvoke()
result = await get_data_async.ainvoke({"url": "https://example.com"})
print("Async execution result:", result)

Async execution result: Data from https://example.com


## 3. Accessing LangGraph State inside Tools via `ToolRuntime``

In advanced agentic workflows, a tool may need access to the current **LangGraph graph state** (e.g., user profile, memory context, active question). `ToolRuntime` allows tools to read state dynamically at runtime.

In [30]:
# Import dependencies for ToolRuntime in LangGraph
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, ToolRuntime
from langchain_core.messages import AIMessage

In [31]:
# Define custom graph state inheriting from MessagesState
class RuntimeState(MessagesState):
    question: str

In [32]:
# Define tool accepting ToolRuntime parameter to read active graph state
@tool
def read_question_from_runtime(runtime: ToolRuntime) -> str:
    """Read the current question from LangGraph state."""
    return runtime.state["question"]

In [33]:
# Node function creating synthetic AIMessage tool call for demonstration
def create_demo_tool_call(state: RuntimeState):
    return {
        "messages": [
            AIMessage(
                content="",
                tool_calls=[{
                    "name": "read_question_from_runtime",
                    "args": {},
                    "id": "demo_call_1",
                    "type": "tool_call",
                }],
            )
        ]
    }

In [34]:
# Instantiate StateGraph using RuntimeState
runtime_builder = StateGraph(RuntimeState)

In [35]:
# Register nodes and connect execution edges
runtime_builder.add_node("create_call", create_demo_tool_call)
runtime_builder.add_node("tools", ToolNode([read_question_from_runtime]))
runtime_builder.add_edge(START, "create_call")
runtime_builder.add_edge("create_call", "tools")
runtime_builder.add_edge("tools", END)

In [36]:
# Compile RuntimeState graph workflow
runtime_graph = runtime_builder.compile()

In [37]:
# Invoke graph with initial state
runtime_result = runtime_graph.invoke({
    "question": "What is LangGraph?",
    "messages": [],
})

In [38]:
# Print tool result retrieved from runtime graph state
print("Tool result:", runtime_result["messages"][-1].content)

Tool result: What is LangGraph?


## 4. Imperative Tool Construction (`Tool` & `StructuredTool`)

Instead of decorators, tools can be constructed imperatively using `Tool(...)` for single string inputs or `StructuredTool.from_function(...)` for multi-argument functions.

In [39]:
# Import Tool class from langchain_core.tools
from langchain_core.tools import Tool

In [40]:
# Define simple single-argument search function
def simple_search_function(query: str) -> str:
    return f"Searching for {query}"

In [41]:
# Construct Tool instance imperatively
simple_search_tool = Tool(
    name="simple_search",
    func=simple_search_function,
    description="Search for information.",
)

In [42]:
# Inspect Tool attributes and invoke with single string argument
print("Name:", simple_search_tool.name)
print("Execution result:", simple_search_tool.invoke("LangGraph"))

Name: simple_search
Execution result: Searching for LangGraph


### 4.1 Single-Input Tools via `Tool.from_function()`

In [43]:
# Define search function for Tool.from_function()
def search_from_function(query: str) -> str:
    return f"Result for {query}"

In [44]:
# Demonstrate Tool.from_function construction
Tool.from_function(
    func=search_from_function,
    name="search_from_function",
    description="Search information.",
)

Tool(name='search_from_function', description='Search information.', func=<function search_from_function at 0x0000019A538B5940>)

In [45]:
# Store Tool.from_function instance in variable
from_function_tool = Tool.from_function(
    func=search_from_function,
    name="search_from_function",
    description="Search information.",
)

In [46]:
# Invoke tool created via Tool.from_function()
print("Execution result:", from_function_tool.invoke("Agentic AI"))

Execution result: Result for Agentic AI


### 4.2 Multi-Input Tools via `StructuredTool`
Use `StructuredTool.from_function()` or `StructuredTool(args_schema=...)` for functions accepting multiple parameters.

In [47]:
# Import StructuredTool and construct tool for multi-argument function
from langchain_core.tools import StructuredTool

def calculate_tax_test(income: float, tax_rate: float) -> float:
    return income * tax_rate

tax_tool_test = StructuredTool.from_function(
    func=calculate_tax_test,
    name="calculate_tax",
    description="Calculate tax from income and tax rate.",
)

print("Args:", tax_tool_test.args)
print("Execution result:", tax_tool_test.invoke({
    "income": 100000,
    "tax_rate": 0.20,
}))

Args: {'income': {'title': 'Income', 'type': 'number'}, 'tax_rate': {'title': 'Tax Rate', 'type': 'number'}}
Execution result: 20000.0


In [48]:
# Define Pydantic schema and construct StructuredTool with explicit schema
class MultiplyInputTest2(BaseModel):
    a: int = Field(description="First number")
    b: int = Field(description="Second number")
    a: int = Field(description="First number")
    b: int = Field(description="Second number")

def direct_multiply(a: int, b: int) -> int:
    return a * b

direct_structured_tool = StructuredTool(
    name="direct_multiply",
    description="Multiply two numbers.",
    func=direct_multiply,
    args_schema=MultiplyInputTest2,
)

print("Execution result:", direct_structured_tool.invoke({
    "a": 12,
    "b": 4,
}))

Execution result: 48


## 5. Enterprise Subclassing of `BaseTool`

For enterprise applications, subclassing `BaseTool` offers maximum control over initialization, custom state properties, sync/async methods (`_run` and `_arun`), and custom error handling.

In [49]:
# Import BaseTool and Type from typing & langchain_core.tools
from typing import Type
from langchain_core.tools import BaseTool

In [50]:
# Define Pydantic input schema for custom BaseTool subclass
class SearchInputTest(BaseModel):
    query: str = Field(description="Search query")

In [51]:
# Subclass BaseTool to create enterprise custom tool class
class MySearchToolTest(BaseTool):
    name: str = "my_search"
    description: str = "Search my custom database."
    args_schema: Type[BaseModel] = SearchInputTest

    def _run(self, query: str) -> str:
        return f"Custom database result for: {query}"

In [52]:
# Instantiate custom BaseTool class
custom_base_tool = MySearchToolTest()

In [53]:
# Execute custom BaseTool invocation
print("Execution result:", custom_base_tool.invoke({
    "query": "LangGraph state management"
}))

Execution result: Custom database result for: LangGraph state management
